In [ ]:
# # Install Keras and tensorflow
# this code will not work because this is older version

from keras.applications.resnet50 import ResNet50, preprocess_input
from keras.preprocessing import image
from scipy.misc import face
import numpy as np

resnet_settings = {"include_top":False, "weights":"imagenet"}
resnet = ResNet50(**resnet_settings)

img = image.array_to_img(face())
img

C:\Users\rahul\AppData\Local\Temp\ipykernel_9824\2886396878.py:5: DeprecationWarning: scipy.misc is deprecated and will be removed in 2.0.0
  from scipy.misc import face


ImportError: cannot import name 'face' from 'scipy.misc' (c:\Users\rahul\AppData\Local\Python\pythoncore-3.11-64\Lib\site-packages\scipy\misc\__init__.py)

In [ ]:
# %pip install pillow
# %pip install pytesseract


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from keras.applications.resnet50 import ResNet50, preprocess_input
from keras.preprocessing import image
#from scipy.misc import face
from keras.utils import load_img, img_to_array
import numpy as np

model = ResNet50(weights="imagenet", include_top=False)
img = load_img("./catt.jpg", target_size=(224,224))
x = img_to_array(img)
x = np.expand_dims(x, axis=0)
x = preprocess_input(x)

features = model.predict(x)
print(features.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
(1, 7, 7, 2048)


In [2]:
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np

resnet = ResNet50(include_top=False, weights="imagenet")

# Replace scipy.misc.face() with your own image
img = image.load_img("catt.jpg")

# Resize
img = img.resize((224, 224))

# Convert to array
x = image.img_to_array(img)

# Add batch dimension
x = np.expand_dims(x, axis=0)

# Preprocess
x = preprocess_input(x)

# Extract features
features = resnet.predict(x)

print(features.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
(1, 7, 7, 2048)


Correct. **That diagram is explaining the architecture, not showing the output of your code.**

Here's how your code relates to the diagram.

### Your code

```python
model = ResNet50(weights="imagenet", include_top=False)

features = model.predict(x)
```

By setting:

```python
include_top=False
```

you're telling Keras:

> "Run the image through the convolutional feature extractor only, and stop before the classifier."

In the diagram, that's equivalent to running only the **orange "deep feature encoder"**.

```
Image
  │
  ▼
c1 → c2 → c3 → c4 → c5
  │
  ▼
features   ←── This is what your code returns
```

The output:

```python
print(features.shape)
```

gave

```text
(1, 7, 7, 2048)
```

Those are the **deep features** produced after the last convolutional layer.

---

### What the diagram shows next

After extracting features, there are two possibilities.

#### 1. Original ImageNet classifier (top path)

```
Image
   │
   ▼
Feature Extractor
   │
   ▼
Fully Connected Layers (f6, f7, f8)
   │
   ▼
1000 ImageNet classes
```

This happens when you use:

```python
model = ResNet50(weights="imagenet", include_top=True)
```

Then the output is class probabilities like:

```
Persian cat
Tiger cat
Lynx
```

---

#### 2. Transfer learning (bottom path)

This is what the figure is emphasizing.

```
Image
   │
   ▼
ResNet Feature Extractor
   │
   ▼
2048-dimensional features
   │
   ▼
Your own classifier
   │
   ▼
Your classes
```

Suppose you're building a classifier for:

* Cats
* Dogs

You don't need the ImageNet classifier.

Instead you use:

```python
features = model.predict(x)
```

and train **your own classifier** (the blue box in the figure) on those features.

---

### Why `include_top=False`?

Because the "top" is this part:

```
f6
f7
f8
```

which was trained for **1000 ImageNet classes**.

If your own task is different, you throw away that top and keep only the feature extractor.

---

### In your code

```python
model = ResNet50(include_top=False)
```

Your output is:

```
Image
   │
   ▼
ResNet
   │
   ▼
(7 × 7 × 2048) feature map   ✅
```

You **do not** get:

```
cat : 98%
dog : 1%
car : 1%
```

because you removed the classifier.

---

This is exactly the idea of **transfer learning**, which is one of the central concepts in the mlcourse.ai deep learning section. The network has already learned useful visual features (edges, textures, object parts, etc.) from ImageNet, and you reuse those features instead of training a new network from scratch.


If there is text on the image, you can read it without unraveling a complicated neural network. For example, check out pytesseract.

In [1]:
import pytesseract
from PIL import Image
import requests
from io import BytesIO

##### Just a random picture from search
img = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTEen2Q6y-M_OqHibt6vM1KmYEAiYI3PXNg7c19nc_8AQq-J16viaAniuc&s=10"
img = requests.get(img)
img = Image.open(BytesIO(img.content))
text = pytesseract.image_to_string(img)

print(text)

Google



It’s good to keep in mind that pytesseract is not a “silver bullet”.

In [2]:
img = "https://habrastorage.org/webt/mj/uv/6o/mjuv6olsh1x9xxe1a6zjy79u1w8.jpeg"
img = requests.get(img)
img = Image.open(BytesIO(img.content))

print(pytesseract.image_to_string(img))

‘BEDROOM
1exI2

DINING AREA
11" 100"

uvinc Room
120" 182"

KITCHEN
102" x T10"

x1t0"


